# LangGraph와 AgentCore 메모리 - 에피소딕 전략 사용

## 소개

이 노트북은 LangGraph 프레임워크를 사용하여 대화형 AI 에이전트에 **에피소딕 메모리 전략**을 활용한 Amazon Bedrock AgentCore 메모리를 통합하는 방법을 보여줍니다. 완전한 대화 세션을 캡처하는 에피소딕 전략에 초점을 맞추어, 에이전트가 특정 식사 계획 에피소드를 회상하고 식이 패턴이 시간에 따라 어떻게 변화하는지 추적할 수 있도록 합니다.

## 튜토리얼 세부사항

| 항목         | 세부내용                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 장기 대화형                                                        |
| 에이전트 사용 사례       | 에피소딕 메모리 전략을 활용한 영양 어시스턴트                               |
| 에이전트 프레임워크   | LangGraph                                                                        |
| LLM 모델           | Anthropic Claude Sonnet 3.7                                                     |
| 튜토리얼 구성요소 | AgentCore 메모리, 에피소딕 전략, LangGraph 훅, 세션 기반 에피소드  |
| 예제 난이도  | 중급                                                                     |

학습 내용:
- 에피소딕 메모리 전략으로 AgentCore 메모리 생성
- 자동 메모리 저장을 위한 pre/post 모델 훅 구현
- 식사 계획 세션을 기억하는 영양 어시스턴트 구축
- 과거 대화 검색 및 반영
- 시간에 따른 식이 패턴 추적

### 시나리오 컨텍스트

이 예제에서는 에피소딕 메모리 전략을 사용하여 완전한 식사 계획 세션을 기억하는 **영양 어시스턴트**를 만듭니다. 에이전트는 레시피 토론, 재료 대체, 식사 피드백을 포함한 전체 대화 에피소드를 캡처합니다. 이를 통해 "지난주에 무엇을 계획했나요?"와 같은 시간 기반 쿼리와 식이 습관의 패턴 분석이 가능합니다.

## 아키텍처

<div style="text-align:left">
    <img src="architecture_episodic.png" width="65%" />
</div>

### 영양 분야에 에피소딕 메모리 전략을 사용하는 이유

- **세션 기반**: 각 식사 계획 대화가 하나의 에피소드입니다
- **시간적 컨텍스트**: 식사가 특정 시간/행사에 연결됩니다
- **패턴 학습**: 선호도가 어떻게 변화하는지 추적합니다
- **풍부한 회상**: 과거 추천의 전체 맥락을 기억합니다

### 에피소딕 메모리 전략 작동 방식

에피소딕 전략은 상호작용을 구조화된 에피소드로 캡처하고 이러한 에피소드를 통해 의미 있는 인사이트를 생성하도록 반영하는 것을 목적으로 설계되었습니다. 이 전략은 무슨 일이 일어났는지뿐만 아니라 각 에피소드의 의도, 생각, 결과도 기록합니다.

#### 에피소딕 전략의 세 단계:

1. **추출** – 단기 메모리에서 유용한 인사이트를 식별하여 메모리 레코드로 장기 메모리에 배치합니다
2. **통합** – 유용한 정보를 새 레코드에 작성할지 기존 레코드에 작성할지 결정합니다
3. **반영** – 에이전트 상호작용의 에피소드를 통해 인사이트가 생성됩니다

#### 전략 출력:

**에피소드** (XML 형식):
- 상황, 의도, 평가, 정당화, 에피소드 수준 반영으로 분류됩니다
- 상호작용이 진행됨에 따라 턴별로 분석됩니다
- 작업 순서와 도구 사용을 이해하는 데 도움이 됩니다

**반영** (백그라운드에서 생성):
- 여러 에피소드를 통합합니다
- 다음을 식별하는 더 넓은 인사이트를 추출합니다:
  - 성공적인 전략과 패턴
  - 잠재적 개선사항
  - 일반적인 실패 모드
  - 여러 상호작용에 걸친 교훈

#### 영양 어시스턴트의 경우:

- **에피소드**: 각 식사 계획 세션 (논의된 레시피, 재료, 결정)
- **반영**: 식이 패턴, 좋아하는 요리, 요리 기술 발전
- **턴별**: 레시피 탐색 → 재료 질문 → 대체 → 최종 선택

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리에 대한 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델 접근 권한

환경 설정부터 시작하겠습니다!

In [ ]:
# https://github.com/langchain-ai/langchain-aws 에서 필요한 라이브러리 설치
%pip install -qr requirements.txt

In [ ]:
import os
import logging

# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
import uuid


region = os.getenv("AWS_REGION", "us-east-1")
logging.getLogger("nutrition-agent").setLevel(logging.DEBUG)

In [ ]:
# 스토어로 사용할 AgentCoreMemoryStore를 가져옵니다
from langgraph_checkpoint_aws import AgentCoreMemoryStore

# 이 예제에서는 컨텍스트 저장을 위해 InMemorySaver만 사용합니다.
# 프로덕션 환경에서는 메모리 스토어와 원활하게 작동하는 AgentCoreMemorySaver를 체크포인터로 사용하는 것을 강력히 권장합니다
# from langgraph_checkpoint_aws import AgentCoreMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from bedrock_agentcore.memory import MemoryClient

In [ ]:
import boto3
import json

# 메모리 실행을 위한 IAM 역할 생성
iam_client = boto3.client("iam")
sts_client = boto3.client("sts")
account_id = sts_client.get_caller_identity()["Account"]

ROLE_NAME = "AgentCoreMemoryExecutionRole"

# AgentCore 메모리를 위한 신뢰 정책 (감마 엔드포인트)
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": [
                    "preprod.genesis-service.aws.internal",
                    "bedrock-agentcore.amazonaws.com",
                    "developer.genesis-service.aws.internal",
                ]
            },
            "Action": "sts:AssumeRole",
        }
    ],
}

# Bedrock 모델 호출을 위한 권한 정책
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                "arn:aws:bedrock:*:*:inference-profile/*",
            ],
        }
    ],
}

try:
    # 기존 역할이 있는지 확인합니다
    role = iam_client.get_role(RoleName=ROLE_NAME)
    MEMORY_EXECUTION_ROLE_ARN = role["Role"]["Arn"]
    print(f"기존 역할 사용: {MEMORY_EXECUTION_ROLE_ARN}")
except iam_client.exceptions.NoSuchEntityException:
    # 역할을 생성합니다
    print(f"IAM 역할 생성 중: {ROLE_NAME}")
    role = iam_client.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for AgentCore Memory with custom strategies",
    )
    MEMORY_EXECUTION_ROLE_ARN = role["Role"]["Arn"]

    # 인라인 정책을 연결합니다
    iam_client.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName="BedrockModelAccess",
        PolicyDocument=json.dumps(permissions_policy),
    )
    print(f"역할 생성 완료: {MEMORY_EXECUTION_ROLE_ARN}")
    print("IAM 전파를 위해 10초 대기 중...")
    import time

    time.sleep(10)

print(f"\n역할 ARN: {MEMORY_EXECUTION_ROLE_ARN}")

In [ ]:
memory_name = "NutritionAssistantEpisodic"
client = MemoryClient(region_name=region)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

override_strategy = {
    "customMemoryStrategy": {
        "name": "NutritionEpisodicExtractor",
        "description": "Nutrition assistant with episodic memory for meal planning insights",
        "namespaces": ["nutrition/{actorId}/{sessionId}/"],
        "configuration": {
            "episodicOverride": {
                "extraction": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Extract meal planning conversations including recipes discussed, ingredients mentioned, dietary considerations, and user feedback.",
                },
                "consolidation": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Consolidate meal planning sessions into episodes, capturing the flow of recipe exploration and decision-making.",
                },
                "reflection": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Generate insights about dietary patterns, favorite recipes, and how meal preferences evolve over time.",
                    "namespaces": ["nutrition/{actorId}/"],
                },
            }
        },
    }
}

memory = client.create_or_get_memory(
    name=memory_name,
    description="Nutrition assistant with episodic memory for meal planning sessions",
    memory_execution_role_arn=MEMORY_EXECUTION_ROLE_ARN,
    strategies=[override_strategy],
)
memory_id = memory["id"]

print(f"에피소딕 메모리 생성 완료: {memory_id}")

### 메모리 설정 개요

AgentCore 에피소딕 메모리 설정에는 다음이 포함됩니다:

- **추출**: 레시피, 재료, 피드백을 포함한 식사 계획 대화를 캡처합니다
- **통합**: 대화를 식사 계획 에피소드로 그룹화합니다
- **반영**: 시간에 따른 식이 패턴과 선호도에 대한 인사이트를 생성합니다
- **네임스페이스**: 사용자별로 에피소드를 구성합니다 (`nutrition/{actorId}/`)

각 대화 세션은 회상하고 분석할 수 있는 에피소드가 됩니다.

## 3단계: 메모리 스토어 및 LLM 초기화

이제 AgentCore 메모리 스토어와 언어 모델을 초기화합니다.

In [ ]:
# 장기 메모리 저장 및 검색을 활성화하기 위해 스토어를 초기화합니다
store = AgentCoreMemoryStore(memory_id=memory_id, region_name=region)

# Bedrock LLM 초기화
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

## 4단계: 메모리 훅 구현

메모리 저장을 자동으로 처리하기 위해 pre 및 post 모델 훅을 생성합니다:

- **Pre 모델 훅**: LLM 호출 전 사용자 메시지를 저장합니다
- **Post 모델 훅**: LLM 호출 후 어시스턴트 응답을 저장합니다

### 메모리 처리 작동 방식

1. 메시지가 actor_id 및 session_id와 함께 AgentCore 메모리에 저장됩니다
2. 에피소딕 전략이 대화를 처리하여 구조화된 에피소드를 생성합니다
3. 에피소드가 턴별 분석과 함께 `nutrition/{actorId}/{sessionId}/` 네임스페이스에 저장됩니다
4. 반영이 에피소드를 통해 생성되어 `nutrition/{actorId}/` 네임스페이스에 저장됩니다
5. 각 에피소드는 상황, 의도, 평가, 대화 흐름을 캡처합니다

**참고**: LangChain 메시지 타입은 에피소드와 반영으로 적절히 처리될 수 있도록 스토어 내부에서 AgentCore 메모리 메시지 타입으로 자동 변환됩니다.

In [ ]:
def pre_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """LLM 호출 전에 실행되어 최신 사용자 메시지를 저장하는 훅"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    # 런타임에 얻는 actor와 session 조합에 메시지를 저장합니다
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # LLM 호출 전 마지막 사용자 메시지를 저장합니다
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    # 에피소딕 전략에서는 메시지만 저장합니다 - 검색이 필요 없습니다
    # 에피소드와 반영은 백그라운드에서 자동으로 생성됩니다
    return {"messages": messages}


def post_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """LLM 호출 후에 실행되어 어시스턴트 응답을 저장하는 훅"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]

    # 런타임에 얻는 actor와 session 조합에 메시지를 저장합니다
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # LLM의 응답을 AgentCore 메모리에 저장합니다
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    return {"messages": messages}

## 5단계: LangGraph 에이전트 생성

이제 메모리 훅이 통합된 LangGraph의 `create_react_agent`를 사용하여 영양 어시스턴트 에이전트를 만듭니다. 도구 노드에는 장기 메모리 검색 도구만 포함되며, pre 및 post 모델 훅이 인수로 지정됩니다.

**참고**: 커스텀 에이전트 구현에서는 이 패턴에 따라 모든 워크플로우에 필요한 대로 Store와 도구를 설정할 수 있습니다. Pre/post 모델 훅을 사용하거나, 전체 대화를 마지막에 저장하는 등의 방식이 가능합니다.

In [ ]:
graph = create_react_agent(
    llm,
    store=store,
    tools=[],  # 이 예제에서는 추가 도구가 필요 없습니다
    checkpointer=InMemorySaver(),  # 대화 상태 관리용
    pre_model_hook=pre_model_hook,  # LLM 호출 전 사용자 메시지를 저장합니다
    post_model_hook=post_model_hook,  # LLM 호출 후 에피소딕 처리를 위해 어시스턴트 응답을 저장합니다
)

## 6단계: 에이전트 런타임 설정

사용자와 세션에 대한 고유 식별자로 에이전트를 설정해야 합니다. 이 ID들은 메모리 구성 및 검색에 중요합니다.

### 그래프 호출 입력
최신 사용자 메시지만 `inputs` 인수로 전달하면 됩니다. 다른 상태 변수도 포함할 수 있지만, 간단한 `create_react_agent`에서는 메시지만 필요합니다.

### LangGraph RuntimeConfig
LangGraph에서 config는 호출 시 필요한 속성(예: 사용자 ID 또는 세션 ID)을 포함하는 `RuntimeConfig`입니다. `AgentCoreMemorySaver`에서는 config에 `thread_id`와 `actor_id`를 설정해야 합니다. 예를 들어, AgentCore 호출 엔드포인트에서 호출자의 ID 또는 사용자 ID를 기반으로 이를 할당할 수 있습니다. 추가 [문서는 여기](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)에서 확인할 수 있습니다.

In [ ]:
actor_id = "user-1"
config = {
    "configurable": {
        "thread_id": "session-1",  # 필수: 내부적으로 Bedrock AgentCore session_id에 매핑됩니다
        "actor_id": actor_id,  # 필수: 내부적으로 Bedrock AgentCore actor_id에 매핑됩니다
    }
}

## 7단계: 에이전트 테스트

음식 선호도에 대한 대화를 통해 영양 어시스턴트를 테스트해 봅시다. 에이전트는 향후 회상 및 패턴 분석을 위해 대화를 에피소드로 자동 캡처합니다.

In [ ]:
# 실행 중 에이전트 출력을 예쁘게 출력하는 헬퍼 함수
def run_agent(query: str, config: RunnableConfig):
    printed_ids = set()
    events = graph.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        stream_mode="values",
    )
    for event in events:
        if "messages" in event:
            for msg in event["messages"]:
                # 이미 출력한 메시지인지 확인합니다
                if id(msg) not in printed_ids:
                    msg.pretty_print()
                    printed_ids.add(id(msg))


# 오늘 밤 좋아하는 식사 중 하나인 연어와 밥, 채소를 요리합니다. 역도 대회 준비를 위한 좋은 매크로 영양소 구성입니다.
# 이 요리의 맛을 더 좋게 하고 단백질과 비타민 섭취를 개선할 수 있는 방법을 물어봅니다.
prompt = """
Hey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has
great macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better
and also improve the protein and vitamins I get?
"""

run_agent(prompt, config)

### 무엇이 저장되었나요?
보시다시피, 모델은 아직 이전 식사 계획 세션에서의 인사이트가 없습니다.

이 pre/post 모델 훅 구현에서는 두 개의 메시지가 저장되었습니다. 사용자의 첫 번째 메시지와 AI 모델의 응답이 모두 AgentCore 메모리에 대화 이벤트로 저장되었습니다. 에피소드와 반영이 생성되기까지 몇 분이 걸릴 수 있으므로, 처음에 아무것도 찾지 못하면 몇 분 후에 다시 시도해 보세요.

이 메시지들은 에피소딕 전략에 의해 처리되어 AgentCore 장기 메모리에 구조화된 에피소드와 반영으로 저장되었습니다. 실제로 스토어를 직접 확인하여 지금까지 저장된 내용을 검증할 수 있습니다:

In [ ]:
# 대화 메시지를 검색합니다
search_namespace = ("nutrition", actor_id, "session-1/")
result = store.search(search_namespace, query="meal", limit=3)
print(f"대화 메시지 결과: {result}")

In [ ]:
# LangGraph에서 에피소딕 장기 메모리를 검색하는 올바른 방법
from bedrock_agentcore.memory import MemoryClient

# 스토어가 아닌 메모리 클라이언트를 직접 사용합니다
memory_client = MemoryClient(region_name=region)

print("=== 장기 에피소딕 메모리 검색 ===")
print(f"메모리 ID: {memory_id}")
print()

# 에피소딕 메모리 (에피소드) 검색
print("1. 에피소딕 네임스페이스: nutrition/user-1/session-1/")
try:
    episodes = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace="nutrition/user-1/session-1/",
        query="meal",
        top_k=3,
    )
    print(f"   {len(episodes)}개의 에피소드 메모리를 찾았습니다")
    for mem in episodes:
        content = mem.get("content", {})
        text = content.get("text", str(content))
        print(f"   - {text[:300]}...")
except Exception as e:
    print(f"   오류: {e}")
print()

# 반영 메모리 검색
print("2. 반영 네임스페이스: nutrition/user-1/")
try:
    reflections = memory_client.retrieve_memories(
        memory_id=memory_id, namespace="nutrition/user-1/", query="meal", top_k=3
    )
    print(f"   {len(reflections)}개의 반영 메모리를 찾았습니다")
    for mem in reflections:
        content = mem.get("content", {})
        text = content.get("text", str(content))
        print(f"   - {text[:300]}...")
except Exception as e:
    print(f"   오류: {e}")

### 에이전트의 스토어 접근

**참고** - AgentCore 메모리는 백그라운드에서 이러한 이벤트를 처리하므로, 메모리가 추출되고 장기 메모리 검색에 임베딩되기까지 몇 분이 걸릴 수 있습니다.

좋습니다! 이전 대화의 메시지를 기반으로 네임스페이스에 장기 메모리가 추출되었음을 확인했습니다.

이제 새 세션을 시작하고 저녁 식사로 무엇을 요리할지 추천을 요청해 봅시다. 에이전트는 스토어를 사용하여 추출된 장기 메모리에 접근하여 사용자가 확실히 좋아할 추천을 제공할 수 있습니다.

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2",  # 새 세션 ID
        "actor_id": actor_id,  # 동일한 actor ID
    }
}

# 새로운 날, 오늘 저녁 무엇을 만들어야 할지 물어봅니다
run_agent("Today's a new day, what should I make for dinner tonight?", config)

### 마무리

보시다시피, 에이전트의 대화는 자동으로 캡처되어 턴별 분석이 포함된 구조화된 에피소드로 처리됩니다. 에피소딕 전략은 여러 식사 계획 세션에 걸쳐 인사이트를 생성하여 패턴을 식별하고 시간에 따른 선호도 변화를 추적합니다.

AgentCoreMemoryStore는 매우 유연하며, pre/post 모델 훅 또는 스토어 작업이 포함된 도구 자체를 포함하여 다양한 방식으로 구현할 수 있습니다. 체크포인팅을 위한 AgentCoreMemorySaver와 함께 사용하면, 전체 대화 상태와 에피소딕 반영을 결합하여 복잡하고 지능적인 에이전트 시스템을 구성할 수 있습니다.